# So sánh kết quả cả nhóm

**Kaggle:** Add Input = output của các notebook `svm_color_histogram` (Luân), `hog_svm` (Nguyên), `resnet18_gradcam` (Quỳnh) (tab *Notebook Output Files*).
`load_results()` tự tìm mọi `result.json` trong `/kaggle/input` và `/kaggle/working/results`,
gộp thành 1 bảng (trùng tên lần chạy thì giữ bản mới nhất).

Std: `auto` = std giữa các seed nếu có > 1 seed (ResNet), ngược lại bootstrap trên tập đánh giá.

In [ ]:
# === SETUP: chạy cell này đầu tiên (Kaggle / Colab / máy) ===
# Lấy code .py theo thứ tự:
#   1. chạy trên máy trong thư mục repo
#   2. code .py đã UPLOAD lên Kaggle (Add Input -> dataset chứa thư mục src/ + configs/) hoặc Colab (/content)
#   3. git clone từ GitHub (Kaggle: bật Settings -> Internet = On)
import os, sys, shutil, subprocess
from pathlib import Path

CODE_SOURCE = "auto"       # "auto" | "upload" | "git"
REPO_URL = "https://github.com/LuanLuKim/cs231-banana-ripeness.git"
BRANCH = "main"
OWNER = "luan"          # tên người chạy, ghi vào bảng kết quả chung

def find_uploaded_code(roots=("/kaggle/input", "/content", "/content/drive/MyDrive"), max_depth=5):
    """Thư mục gốc của code đã upload: thư mục chứa package src/ (src/__init__.py + src/experiment.py)."""
    skip = {"images", "masks", "results", "cache", "reports", ".git", "drive", "sample_data", "__pycache__"}
    for base in map(Path, roots):
        if not base.exists():
            continue
        for dirpath, dirnames, filenames in os.walk(base):
            d = Path(dirpath)
            if d.name == "src" and "__init__.py" in filenames and "experiment.py" in filenames:
                return d.parent
            depth = len(d.parts) - len(base.parts)
            dirnames[:] = [] if depth >= max_depth else sorted(x for x in dirnames if x not in skip)
    return None

local = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "__init__.py").exists()), None)
uploaded = find_uploaded_code() if CODE_SOURCE in ("auto", "upload") and local is None else None
if local is not None and CODE_SOURCE == "auto":
    CODE_DIR, source = local, "máy"
elif uploaded is not None:
    # chép sang /tmp: thư mục /kaggle/input chỉ đọc, và để output notebook không chứa code
    CODE_DIR = Path("/tmp/cs231-code")
    shutil.rmtree(CODE_DIR, ignore_errors=True)
    shutil.copytree(uploaded, CODE_DIR, ignore=shutil.ignore_patterns("__pycache__", ".git", "*.ipynb"))
    source = f"upload ({uploaded})"
elif CODE_SOURCE == "upload":
    raise FileNotFoundError("Không thấy code đã upload: Add Input dataset chứa thư mục src/ và configs/")
else:
    CODE_DIR = Path("/tmp/cs231-banana-ripeness")
    if (CODE_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(CODE_DIR), "pull", "-q", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO_URL, str(CODE_DIR)], check=True)
    commit = subprocess.run(["git", "-C", str(CODE_DIR), "rev-parse", "--short", "HEAD"],
                            capture_output=True, text=True).stdout.strip()
    source = f"git {commit}"
sys.path.insert(0, str(CODE_DIR))      # thư mục gốc repo -> import src.*
for m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[m]        # chạy lại cell sau khi đổi code -> nạp bản mới
os.environ["BANANA_OWNER"] = OWNER
print("Code:", CODE_DIR, "| nguồn:", source)
if not (CODE_DIR / "configs").exists():
    print("!! Thiếu thư mục configs/ (cần configs/kaggle.yaml, configs/experiments.yaml) — upload cả thư mục configs")

%load_ext autoreload
%autoreload 2

In [ ]:
from src.experiment import load_results, compare_table, plot_comparison, summarize_runs
import pandas as pd
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)
res = load_results()        # hoặc load_results(["/kaggle/input/<notebook-10>", "/kaggle/input/<notebook-20>", ...])
print(len(res), "dòng |", res.run_name.nunique(), "lần chạy")
res.groupby(["owner", "model"]).run_name.nunique()

## Bảng chính cho slide 8

In [ ]:
REPORTS = Path("/kaggle/working/reports") if Path("/kaggle/working").exists() else Path("reports")
REPORTS.mkdir(parents=True, exist_ok=True)
res.to_csv(REPORTS / "results_all_runs.csv", index=False)
print("=== mixed — test (3 dataset) ===")
display(compare_table(res, split="test", scheme="mixed", save_path=REPORTS / "results_mixed_test.csv"))
print("=== external — kiểm định ngoài BananaImageBD ===")
display(compare_table(res, split="external", scheme="external", save_path=REPORTS / "results_external.csv"))

In [ ]:
plot_comparison(res, split="test", scheme="mixed", save_path=REPORTS / "compare_mixed_test.png");
plot_comparison(res, split="external", scheme="external", save_path=REPORTS / "compare_external.png");

## Mô hình tốt nhất của từng người (chọn theo val)

In [ ]:
from src.experiment import best_run
import matplotlib.pyplot as plt
models = sorted(res.model.unique())
for scheme, split in [("mixed", "test"), ("external", "external")]:
    fig, axes = plt.subplots(1, len(models), figsize=(4.6 * len(models), 4), squeeze=False)
    for ax, m in zip(axes[0], models):
        try:
            best_run(res, scheme=scheme, by_split="val", model=m).plot_confusion(split, ax=ax)
        except ValueError:
            ax.axis("off")
    fig.suptitle(f"{scheme} / {split}"); plt.tight_layout(); plt.show()

## Macro-F1 theo từng dataset trong test (mixed)

In [ ]:
s = summarize_runs(res)
cols = [c for c in s.columns if c.startswith("f1_macro__")]
s[(s.split == "test") & (s.scheme == "mixed")].set_index(["model", "tag"])[["f1_macro", *cols]] \
    .sort_values("f1_macro", ascending=False).round(3)

## F1 từng lớp (Half_Ripe ↔ Ripe là cặp khó)

In [ ]:
cols = [f"f1_{c}" for c in ["Unripe", "Half_Ripe", "Ripe", "Overripe"]]
s[(s.split == "test") & (s.scheme == "mixed")].set_index(["model", "tag"])[cols] \
    .sort_values("f1_Half_Ripe", ascending=False).round(3)